In [4]:
#Create the main Repo list
import os
import pandas as pd
from urllib.parse import urlparse

# === INPUT / OUTPUT ===
SRC_CSV = r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\Clone_Status.csv"
OUT_DIR = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet"
OUT_CSV = os.path.join(OUT_DIR, "3.2_Total_Repo.csv")
os.makedirs(OUT_DIR, exist_ok=True)

# === Load ===
df = pd.read_csv(SRC_CSV, dtype=str).fillna("")
df.columns = [c.strip() for c in df.columns]

# --- Find columns ---
def pick_col(candidates, cols):
    cols_lower = {c.lower(): c for c in cols}
    for cand in candidates:
        if cand.lower() in cols_lower:
            return cols_lower[cand.lower()]
    return None

clone_col = pick_col(["clone_status"], df.columns)
yml_col   = pick_col(["yml_detected"], df.columns)
url_col   = pick_col(["html_url"], df.columns)

if not clone_col or not yml_col or not url_col:
    missing = [name for name, col in {"clone_status": clone_col, "yml_detected": yml_col, "html_url/htm_url": url_col}.items() if not col]
    raise ValueError(f"Missing required column(s): {', '.join(missing)}")

# --- Normalize "yes" detection ---
def is_yes(x: str) -> bool:
    return str(x).strip().lower() in {"yes", "true", "y", "1"}

filtered = df[ df[clone_col].apply(is_yes) & df[yml_col].apply(is_yes) ].copy()

# --- Build full_name = owner.repo ---
def url_to_full_name(u: str) -> str:
    try:
        path = urlparse(str(u).strip()).path.strip("/")
        if not path:
            return ""
        if path.endswith(".git"):
            path = path[:-4]
        parts = path.split("/")
        if len(parts) >= 2:
            return f"{parts[0]}.{parts[1]}"
        return path
    except Exception:
        return ""

# Ensure full_name is lowercase
filtered["full_name"] = filtered[url_col].apply(url_to_full_name).str.lower()

# --- Keep only URL + full_name ---
out = filtered[[url_col, "full_name"]].rename(columns={url_col: "html_url"})

# Drop duplicates
out = out.drop_duplicates(subset=["html_url"]).reset_index(drop=True)

# Save
out.to_csv(OUT_CSV, index=False)
print(f"Saved: {OUT_CSV} (rows={len(out)})")


Saved: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet\3.2_Total_Repo.csv (rows=4518)


In [5]:
#Adds the Instru_tests for each repo

import os
import pandas as pd
from collections import defaultdict

# === PATHS ===
REPO_CSV = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet\3.2_Total_Repo.csv"
TEST_DIR = r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\All_Test_Files"
OUT_CSV  = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet\3.2_Total_Repo.csv"

# === Load repo list ===
repos = pd.read_csv(REPO_CSV, dtype=str).fillna("")
if "full_name" not in repos.columns:
    raise ValueError("Expected column 'full_name' in 3.1_Total_Repo.csv (format: owner.repo).")
repos["full_name"] = repos["full_name"].astype(str).str.strip()

# === Detection keywords ===
INSTRU_HINTS_NATIVE  = [
    "instrumentation", "androidtest", "connectedandroidtest",
    "espresso", "uiautomator", "orchestrator", "manageddevices", "gmd"
]
INSTRU_HINTS_FLUTTER = [
    "flutter", "dart"
]

def is_instru_file(fname: str) -> bool:
    name = fname.lower()
    return any(k in name for k in INSTRU_HINTS_NATIVE + INSTRU_HINTS_FLUTTER)

def classify_test(fname: str) -> str:
    lname = fname.lower()
    if any(k in lname for k in INSTRU_HINTS_FLUTTER):
        return "flutter"
    if any(k in lname for k in INSTRU_HINTS_NATIVE):
        return "native"
    return ""

# === Count tests per repo ===
native_counts  = defaultdict(int)
flutter_counts = defaultdict(int)

for fname in os.listdir(TEST_DIR):
    fpath = os.path.join(TEST_DIR, fname)
    if not os.path.isfile(fpath):
        continue
    if "__" not in fname:
        continue

    repo_token = fname.split("__", 1)[0].strip()
    if not repo_token:
        continue
    if not is_instru_file(fname):
        continue

    kind = classify_test(fname)
    if kind == "flutter":
        flutter_counts[repo_token] += 1
    elif kind == "native":
        native_counts[repo_token] += 1

# === Merge into repo DataFrame ===
repos["native_instru_test"]  = repos["full_name"].map(lambda k: native_counts.get(k, 0)).astype(int)
repos["flutter_instru_test"] = repos["full_name"].map(lambda k: flutter_counts.get(k, 0)).astype(int)
repos["Intru_test"] = (repos["native_instru_test"] + repos["flutter_instru_test"] > 0)

# === Save ===
repos.to_csv(OUT_CSV, index=False)
print(f"Saved: {OUT_CSV} (rows={len(repos)})")


Saved: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet\3.2_Total_Repo.csv (rows=4518)


the following cell is adjusted to includes two more new column created in yaml v4.0 for detecting flutter signal and device

In [6]:
# Aggregate instru_t_ci_signal and per-platform YML counts, append to main (case-insensitive)
import os
import re
import pandas as pd

BASE_DIR = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet"
MAIN_CSV = os.path.join(BASE_DIR, "3.2_Total_Repo.csv")
YML_CSV  = os.path.join(BASE_DIR, "3.1.1_YML_Files_V4.0.csv")
OUT_CSV  = MAIN_CSV  # overwrite

def normalize_name(s: pd.Series) -> pd.Series:
    return s.astype(str).str.strip().str.lower()

def sanitize_col(name: str) -> str:
    s = re.sub(r"\W+", "_", str(name).strip().lower())
    s = re.sub(r"_+", "_", s).strip("_")
    return s or "unknown"

def to_bool_series(s: pd.Series) -> pd.Series:
    truthy = {"true","1","yes","y","t"}
    falsy  = {"false","0","no","n","f"}
    s = s.astype(str).str.strip().str.lower()
    return s.map(lambda x: True if x in truthy else (False if x in falsy else False)).astype("boolean")

def dedup_preserve(seq):
    seen, out = set(), []
    for x in seq:
        x = str(x)
        if x and x not in seen:
            seen.add(x); out.append(x)
    return out

# --- Load main ---
if not os.path.isfile(MAIN_CSV):
    raise FileNotFoundError(f"Main CSV not found: {MAIN_CSV}")
main = pd.read_csv(MAIN_CSV, dtype=str).fillna("")
if "full_name" not in main.columns:
    raise ValueError("Main CSV must contain 'full_name'.")
main["full_name_lc"] = normalize_name(main["full_name"])

# --- Load YML ---
if not os.path.isfile(YML_CSV):
    raise FileNotFoundError(f"YML CSV not found: {YML_CSV}")
yml = pd.read_csv(YML_CSV, dtype=str).fillna("")
for col in ("full_name", "ci_platform"):
    if col not in yml.columns:
        raise ValueError(f"{os.path.basename(YML_CSV)} must contain '{col}'.")

# ensure expected columns exist
for col, default in (
    ("instru_t_ci_signal", ""),
    ("confidence", ""),
    ("confidence_reason", ""),
    # NEW: Flutter columns expected from detector; create safe defaults if absent
    ("flutter_integ_t_signal", "false"),
    ("flutter_integ_t_d", ""),
):
    if col not in yml.columns:
        yml[col] = default

yml["full_name_lc"] = normalize_name(yml["full_name"])

# --- Aggregate per-platform YML counts and totals ---
plat_counts = (
    yml.groupby(["full_name_lc", "ci_platform"])
       .size()
       .unstack(fill_value=0)
)
if not plat_counts.empty:
    plat_counts = plat_counts.rename(columns={c: sanitize_col(c) for c in plat_counts.columns})
    # collapse potential duplicates caused by sanitation
    plat_counts = plat_counts.T.groupby(level=0).sum().T
    plat_counts = plat_counts.reset_index()
else:
    plat_counts = pd.DataFrame(columns=["full_name_lc"])

total_counts = (
    yml.groupby("full_name_lc").size().rename("Total_YMLs").reset_index()
)

# --- Aggregate instru_t_ci_signal (True if any row True) ---
yml["instru_t_ci_signal"] = to_bool_series(yml["instru_t_ci_signal"])
agg_flag = (
    yml[["full_name_lc", "instru_t_ci_signal"]]
      .groupby("full_name_lc", as_index=False)["instru_t_ci_signal"]
      .any()
)
agg_flag["instru_t_ci_signal"] = agg_flag["instru_t_ci_signal"].astype("boolean")

# --- Aggregate confidence + confidence_reason per repo ---
yml["conf_norm"] = yml["confidence"].str.strip().str.lower()
def aggregate_confidence(group: pd.DataFrame) -> pd.Series:
    confs = set(group["conf_norm"].dropna().tolist())
    if "high" in confs:
        level = "high"
    elif "medium" in confs:
        level = "medium"
    elif "low" in confs:
        level = "low"
    else:
        level = ""

    if level:
        reasons = group.loc[group["conf_norm"] == level, "confidence_reason"].astype(str).str.strip()
        reasons = [r for r in reasons if r]
        agg_reason = " || ".join(dedup_preserve(reasons))
    else:
        agg_reason = ""

    return pd.Series({"instru_t_ci_confidence": level, "confidence_reason": agg_reason})

agg_conf = (
    yml.groupby("full_name_lc")
       .apply(aggregate_confidence)
       .reset_index()
)

# --- NEW: Aggregate Flutter columns ---
# repo-level flutter_integ_t_signal: True if any row True
yml["flutter_integ_t_signal"] = to_bool_series(yml["flutter_integ_t_signal"])
flutter_sig = (
    yml.groupby("full_name_lc", as_index=False)["flutter_integ_t_signal"]
       .any()
)
flutter_sig["flutter_integ_t_signal"] = flutter_sig["flutter_integ_t_signal"].astype("boolean")

# Device token normalization map and nulls
ALIASES_DEV = {
    # windows
    "win": "windows", "windows-latest": "windows", "windows-2019": "windows", "windows-2022": "windows",
    # linux
    "ubuntu": "linux", "ubuntu-20.04": "linux", "ubuntu-22.04": "linux", "debian": "linux", "linux-latest": "linux",
    # mac
    "osx": "macos", "darwin": "macos", "macos-latest": "macos", "macos-13": "macos", "macos-14": "macos",
    # ios
    "iphoneos": "ios", "apple-ios": "ios",
    # android (+ common typo)
    "android-os": "android", "andorid": "android",
    # web
    "chrome": "web", "browser": "web",
}
NULS = {"", "nan", "none", "null"}

# repo-level flutter_integ_t_d: tokenize, normalize, dedupe, SORT, and join
yml["flutter_integ_t_d_norm"] = yml["flutter_integ_t_d"].astype(str).str.strip().str.lower()

SEP_RE = re.compile(r"[,\s;/|/]+")

def tokenize_devices(cell: str):
    if not cell or cell in NULS:
        return []
    toks = [t.strip().lower() for t in SEP_RE.split(cell) if t.strip()]
    normed = []
    for t in toks:
        t = ALIASES_DEV.get(t, t)
        # normalize substrings that contain "android"
        if "andorid" in t: t = "android"
        if "android" in t: t = "android"
        normed.append(t)
    return normed

def agg_flutter_devices(group: pd.DataFrame) -> pd.Series:
    bag = set()
    for v in group["flutter_integ_t_d_norm"]:
        for tok in tokenize_devices(v):
            if tok not in NULS:
                bag.add(tok)
    # Sort alphabetically; join with comma+space
    out_list = sorted(bag)
    return pd.Series({"flutter_integ_t_d": ", ".join(out_list)})

flutter_dev = (
    yml.groupby("full_name_lc")
       .apply(agg_flutter_devices)
       .reset_index()
)

# --- Merge aggregates together ---
agg = (total_counts
       .merge(plat_counts, on="full_name_lc", how="left")
       .merge(agg_flag,    on="full_name_lc", how="left")
       .merge(agg_conf,    on="full_name_lc", how="left")
       .merge(flutter_sig, on="full_name_lc", how="left")    # NEW
       .merge(flutter_dev, on="full_name_lc", how="left"))   # NEW

# --- Merge into main (case-insensitive) ---
out = main.merge(agg, on="full_name_lc", how="left").drop(columns=["full_name_lc"])

# Normalize numeric/platform columns to int (fill NaN with 0)
platform_cols = [c for c in plat_counts.columns if c != "full_name_lc"]
for c in platform_cols:
    if c not in out.columns:
        out[c] = 0
    out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0).astype(int)

if "Total_YMLs" not in out.columns:
    out["Total_YMLs"] = 0
out["Total_YMLs"] = pd.to_numeric(out["Total_YMLs"], errors="coerce").fillna(0).astype(int)

# Final instru_t_ci_signal in main: aggregated value
existing = out["instru_t_ci_signal"] if "instru_t_ci_signal" in out.columns else pd.Series([False]*len(out), index=out.index)
existing = to_bool_series(existing).fillna(False)
yml_flag = out.get("instru_t_ci_signal", pd.Series([False]*len(out), index=out.index))
yml_flag = to_bool_series(yml_flag).fillna(False)
out["instru_t_ci_signal"] = yml_flag.astype("boolean")

# --- Ensure Flutter columns exist with correct types/defaults ---
if "flutter_integ_t_signal" not in out.columns:
    out["flutter_integ_t_signal"] = False
out["flutter_integ_t_signal"] = to_bool_series(out["flutter_integ_t_signal"]).fillna(False).astype("boolean")

if "flutter_integ_t_d" not in out.columns:
    out["flutter_integ_t_d"] = ""
out["flutter_integ_t_d"] = out["flutter_integ_t_d"].astype(str).fillna("").str.strip()

# Save
out.to_csv(OUT_CSV, index=False)
print(f"Saved: {OUT_CSV} (rows={len(out)})")
print(f"instru_t_ci_signal True={int(out['instru_t_ci_signal'].sum())} / {len(out)}")
print("Platform columns added:", [c for c in platform_cols if c in out.columns])
print(f"flutter_integ_t_signal True={int(out['flutter_integ_t_signal'].sum())} / {len(out)}")


C:\Users\gilla\AppData\Local\Temp\ipykernel_13720\102427299.py:114: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(aggregate_confidence)


Saved: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet\3.2_Total_Repo.csv (rows=4518)
instru_t_ci_signal True=449 / 4518
Platform columns added: ['appveyor', 'azure_pipelines', 'bitbucket', 'bitrise', 'circle_ci', 'cirrus', 'codemagic', 'github_actions', 'gitlab', 'semaphore', 'travis_ci']
flutter_integ_t_signal True=35 / 4518


C:\Users\gilla\AppData\Local\Temp\ipykernel_13720\102427299.py:174: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(agg_flutter_devices)


In [7]:
# adds the instru testing signals config and unit test signals both CI and Configs

import os
import pandas as pd

BASE_DIR = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet"

FILES = {
    "instru_t_signal_config": ("3.1.1_Instru_T_Signal_Config.csv",
                               ["instru_t_signal_config", "instru_test_signal_config"]),
    "unit_t_signal_ci":       ("3.1.2_Unit_T_Signal_CI.csv",
                               ["unit_t_signal_ci", "unit_test_signal_ci"]),
    "unit_t_signal_config":   ("3.1.2_Unit_T_Signal_Config.csv",
                               ["unit_t_signal_config", "unit_test_config_signal"]),
}

MAIN_CSV = os.path.join(BASE_DIR, "3.2_Total_Repo.csv")
OUT_CSV  = MAIN_CSV  # overwrite

def normalize_name(s: pd.Series) -> pd.Series:
    return s.astype(str).str.strip().str.lower()

def to_bool_series(s: pd.Series) -> pd.Series:
    truthy = {"true","1","yes","y","t"}
    falsy  = {"false","0","no","n","f"}
    s = s.astype(str).str.strip().str.lower()
    return s.map(lambda x: True if x in truthy else (False if x in falsy else False)).astype("boolean")

def load_and_aggregate_bool(path: str, out_col: str, candidates: list[str]) -> pd.DataFrame:
    if not os.path.isfile(path):
        print(f"[WARN] Missing file -> {os.path.basename(path)}; skipping {out_col}.")
        return pd.DataFrame(columns=["full_name", out_col])
    df = pd.read_csv(path, dtype=str).fillna("")
    if "full_name" not in df.columns:
        print(f"[WARN] 'full_name' not in {os.path.basename(path)}; skipping {out_col}.")
        return pd.DataFrame(columns=["full_name", out_col])

    df["full_name"] = normalize_name(df["full_name"])
    flag_col = next((c for c in candidates if c in df.columns), None)
    if flag_col is None:
        print(f"[WARN] None of {candidates} found in {os.path.basename(path)}; skipping {out_col}.")
        return pd.DataFrame(columns=["full_name", out_col])

    b = to_bool_series(df[flag_col])
    agg = (
        pd.DataFrame({"full_name": df["full_name"], out_col: b})
          .groupby("full_name", as_index=False)[out_col]
          .any()
    )
    agg[out_col] = agg[out_col].astype("boolean")
    return agg

# --- Load & normalize main ---
if not os.path.isfile(MAIN_CSV):
    raise FileNotFoundError(f"Main CSV not found: {MAIN_CSV}")

main = pd.read_csv(MAIN_CSV, dtype=str).fillna("")
if "full_name" not in main.columns:
    raise ValueError("Main CSV must contain a 'full_name' column.")
main["full_name"] = normalize_name(main["full_name"])

# --- Merge aggregated flags (default False; True if any repo hit) ---
for out_col, (fname, aliases) in FILES.items():
    agg_path = os.path.join(BASE_DIR, fname)
    agg = load_and_aggregate_bool(agg_path, out_col, aliases)

    # ensure column exists in main with default False (nullable boolean)
    if out_col not in main.columns:
        main[out_col] = pd.Series([False] * len(main), index=main.index, dtype="boolean")
    else:
        # coerce any existing values safely -> boolean, unknowns -> False
        main[out_col] = to_bool_series(main[out_col])

    if not agg.empty:
        lookup = agg.set_index("full_name")[out_col]
        upd = main["full_name"].map(lookup).astype("boolean").fillna(False)
        # OR (True wins)
        main[out_col] = (main[out_col] | upd).astype("boolean")

# --- Save ---
main.to_csv(OUT_CSV, index=False)
print(f"Saved: {OUT_CSV} (rows={len(main)})")

# Optional: quick counts
for col in FILES.keys():
    if col in main.columns:
        print(f"{col}: True={int(main[col].sum())} / {len(main)}")


Saved: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet\3.2_Total_Repo.csv (rows=4518)
instru_t_signal_config: True=2280 / 4518
unit_t_signal_ci: True=2162 / 4518
unit_t_signal_config: True=2303 / 4518


In [8]:
# -*- coding: utf-8 -*-
"""
RQ1 Sections 5 & 6 grouping with robust column resolution and EXPLICIT FLIP rule:
- Follow instru_t_ci_signal
- Flip True -> False only if:
    (flutter_integ_t_d is non-blank AND does NOT include 'android'/'andorid')
    AND (confidence_reason lacks Android runtime keywords after stripping the generic boost)

Adds ONLY:
  - Android_Instu_Test (bool)
  - Readiness_Level (High/Medium/Low/None)
  - Cohort_4way (Both/CI-only/Build-only/None)
  - Adoption_TestBacked (bool)
  - RQ1_Approach
  - Android_Env_Family
"""

import re
import pandas as pd
from pathlib import Path

# === PATH ===
CSV_PATH = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet\3.2_Total_Repo.csv")

# ---------- Utilities ----------
def to_bool(x):
    if pd.isna(x): return None
    s = str(x).strip().lower()
    if s in {"1","true","t","yes","y"}: return True
    if s in {"0","false","f","no","n"}: return False
    return None

def norm(s: str) -> str:
    """Normalize a column name for matching."""
    return re.sub(r'[_\s\-.]+', '', str(s).strip().lower())

def contains_all(s: str, tokens) -> bool:
    s2 = norm(s)
    return all(t in s2 for t in tokens)

# Preferred aliases (normalized)
ALIASES = {
    "instru_t_ci_signal":       ["instru_t_ci_signal","instru_ci_signal","instru_t_ci","ci_signal","yaml_pred"],
    "instru_t_signal_config":   ["instru_t_signal_config","build_signal","build_pred"],
    "Intru_test":               ["intru_test","at_pred","androidtest_present"],
    "flutter_integ_t_signal":   ["flutter_integ_t_signal","flutter_cli_signal","flutter_cli","flutter_signal"],
    "flutter_integ_t_d":        ["flutter_integ_t_d","flutter_android","flutter_on_android","flutter_devices","flutter_device_list"],
    "confidence_reason":        ["confidence_reason","reason_at_clone","detector_reason","ci_reason","ci_confidence_reason"],
    "full_name":                ["full_name","repo_full_name","name"],
    "html_url":                 ["html_url","repo_url","url"],
}

# Heuristic token sets (normalized, without punctuation)
HEURISTICS = {
    "instru_t_ci_signal":     [["instru","ci"], ["yaml","pred"]],
    "instru_t_signal_config": [["instru","signal","config"], ["build","pred"]],
    "Intru_test":             [["intru","test"], ["android","test"]],
    "flutter_integ_t_signal": [["flutter","signal"], ["flutter","cli"]],
    "flutter_integ_t_d":      [["flutter","d"], ["flutter","device"]],
    "confidence_reason":      [["confidence","reason"], ["detector","reason"], ["ci","reason"]],
    "full_name":              [["full","name"]],
    "html_url":               [["html","url"], ["repo","url"]],
}

def resolve_col(df: pd.DataFrame, key: str) -> str:
    """Resolve a column using aliases, case-insensitive normalization, then heuristics."""
    cols = list(df.columns)
    nmap = {norm(c): c for c in cols}

    # 1) exact normalized alias match
    for alias in ALIASES.get(key, []):
        a = norm(alias)
        if a in nmap:
            return nmap[a]

    # 2) direct normalized column name equals normalized key
    k = norm(key)
    if k in nmap:
        return nmap[k]

    # 3) heuristic token match
    for cand in cols:
        for token_set in HEURISTICS.get(key, []):
            if contains_all(cand, token_set):
                return cand

    # 4) helpful error with suggestions
    raise KeyError(
        f"Required column not found for '{key}'. "
        f"Tried aliases={ALIASES.get(key, [])} and heuristics={HEURISTICS.get(key, [])}. "
        f"Available columns: {cols}"
    )

# ---------- Runtime detectors (strip generic boost; then look for runtime cues) ----------
RX_AT_BOOST = re.compile(r'boosted\s*\(\s*androidtest\s+present\s*\)', re.I)

# Android runtime keywords (examples + concrete tasks/tools)
RX_RUNTIME = re.compile(
    r'(?:gcloud\s+firebase|flank|saucectl|appcenter\s+test|maestro|browserstack|bstack|'
    r'connected(?:android)?test|connectedcheck|devicecheck|'
    r'(?:^|[\s:/.-])(?:gradlew?|gradle)\b|'
    r'\bandroidtest\b)',
    re.I
)

def runtime_keywords_absent(reason: str) -> bool:
    """True if (after stripping generic boost) there are NO runtime keywords."""
    r = RX_AT_BOOST.sub("", str(reason or ""))
    return not bool(RX_RUNTIME.search(r))

# ---------- Devices helpers ----------
NULS = {"", "nan", "none", "null"}
SEP_RE = re.compile(r"[,\s;/|/]+")

def devices_nonblank_and_nonandroid(cell) -> bool:
    """
    True iff flutter_integ_t_d is non-blank AND contains no 'android'/'andorid' token.
    """
    if pd.isna(cell):
        return False
    s = str(cell).strip().lower()
    if s in NULS:
        return False
    toks = [t.strip() for t in SEP_RE.split(s) if t.strip()]
    if not toks:
        return False
    # if any token contains 'android' or the common typo 'andorid' -> not nonandroid
    contains_android = any(("android" in t) or ("andorid" in t) for t in toks)
    return not contains_android

def has_android_in_devices(cell) -> bool:
    """Used later for approach labeling."""
    if pd.isna(cell): return False
    s = str(cell).strip().lower()
    return ("android" in s) or ("andorid" in s)

# ---------- Environment & Approach (for RQ1 outputs) ----------
RX_3P   = re.compile(r'\b(?:gcloud\s+firebase|flank|saucectl|appcenter\s+test|maestro|browserstack|bstack)\b', re.I)
RX_GMD  = re.compile(r'\bmanaged(?:virtual)?device\b|\bgmd\b', re.I)
RX_ADB  = re.compile(r'\bam\s+instrument\b', re.I)
RX_EMULATOR = re.compile(
    r'emulator\s+-avd|reactivecircus/android-emulator-runner|android-wait-for-emulator|'
    r'\bavdmanager\b|sdkmanager[^"\n]*system-images|circle-android wait-for-boot|start-emulator\.sh',
    re.I
)
RX_GRADLE_CONNECTED = re.compile(
    r'(?:^|[\s:/.-])(?:gradlew?\s+\S*\s*)?(?:connected(?:Android)?Test|connectedCheck|deviceCheck)\b', re.I
)
RX_GRADLE_ANDROIDTEST_TASK = re.compile(
    r'(?:^|[\s:/.-])(?:gradlew?\s+\S*\s*)?(?:(?:\S*:)+)?androidTest\b', re.I
)

def classify_env_family(reason: str) -> str:
    r = str(reason or "")
    if RX_3P.search(r):           return "3P Lab"
    if RX_GMD.search(r):          return "GMD"
    if RX_EMULATOR.search(r):     return "Emulator"
    if RX_ADB.search(r):          return "Real Device"
    return "Unknown"

def classify_approach(reason: str, ci_flag: bool, android_ci: bool, flt_sig: bool, flt_on_android: bool) -> str:
    r = str(reason or "")
    if not ci_flag:
        return "No CI"
    if android_ci:
        if RX_3P.search(r):       return "3P CLI"
        if RX_ADB.search(r):      return "ADB"
        if RX_GMD.search(r):      return "Gradle (GMD)"
        if RX_GRADLE_CONNECTED.search(r) or RX_GRADLE_ANDROIDTEST_TASK.search(r): return "Gradle"
        if flt_on_android:        return "Flutter CLI (Android)"
        return "CI non-Android/Unknown"
    else:
        if flt_sig and not flt_on_android:
            return "Flutter CLI (non-Android)"
        return "CI non-Android/Unknown"

# ---------- Main ----------
def main():
    if not CSV_PATH.exists():
        raise FileNotFoundError(f"File not found: {CSV_PATH}")

    df = pd.read_csv(CSV_PATH, low_memory=False)

    # Resolve columns robustly
    col_ci      = resolve_col(df, "instru_t_ci_signal")
    col_build   = resolve_col(df, "instru_t_signal_config")
    col_tests   = resolve_col(df, "Intru_test")
    col_flt_sig = resolve_col(df, "flutter_integ_t_signal")
    col_flt_d   = resolve_col(df, "flutter_integ_t_d")
    col_reason  = resolve_col(df, "confidence_reason")

    # Normalize booleans (flutter_integ_t_d remains a string list)
    for c in [col_ci, col_build, col_tests, col_flt_sig]:
        df[c] = df[c].apply(to_bool)

    # --- Follow CI and only FLIP True->False under strict conditions ---
    ci_true = df[col_ci].apply(lambda x: True if x is True else False)

    nonblank_nonandroid = df[col_flt_d].apply(devices_nonblank_and_nonandroid)
    no_runtime_keywords = df[col_reason].apply(runtime_keywords_absent)

    # Flip condition: devices list is explicitly non-Android AND no runtime keywords
    flip = nonblank_nonandroid & no_runtime_keywords

    # Android_Instu_Test: follow CI, flip only when condition holds
    df["Android_Instu_Test"] = (ci_true & (~flip)).astype(bool)

    # ----- Section 5: Readiness + Cohort + Adoption -----
    def cohort(ci, build):
        ci_b, b_b = to_bool(ci), to_bool(build)
        if ci_b is True and b_b is True:        return "Both (CI & Build)"
        if ci_b is True and b_b is not True:    return "CI-only"
        if ci_b is not True and b_b is True:    return "Build-only"
        return "None"
    df["Cohort_4way"] = [cohort(ci, b) for ci, b in zip(df[col_ci], df[col_build])]

    def readiness(android_ci, ci, build):
        if android_ci is True:                   return "High"
        ci_b, b_b = to_bool(ci), to_bool(build)
        if b_b is True and (ci_b is not True):   return "Medium"
        if ci_b is True and (b_b is not True):   return "Low"
        return "None"
    df["Readiness_Level"] = [readiness(a, ci, b) for a, ci, b in zip(df["Android_Instu_Test"], df[col_ci], df[col_build])]

    df["Adoption_TestBacked"] = (df["Android_Instu_Test"] == True) & (df[col_tests] == True)

    # ----- Section 6: Environment + Approach -----
    df["Android_Env_Family"] = df[col_reason].apply(classify_env_family)
    df["RQ1_Approach"] = [
        classify_approach(
            reason,
            (to_bool(ci_flag) is True),
            (a_ci is True),
            (to_bool(flt_sig) is True),
            (has_android_in_devices(flt_d) is True)
        )
        for reason, ci_flag, a_ci, flt_sig, flt_d in zip(
            df[col_reason], df[col_ci], df["Android_Instu_Test"], df[col_flt_sig], df[col_flt_d]
        )
    ]

    # Save
    df.to_csv(CSV_PATH, index=False, encoding="utf-8-sig")
    print("[OK] Updated file in place:", CSV_PATH)

    # Quick summary
    try:
        N = len(df)
        high = int((df["Readiness_Level"]=="High").sum())
        med  = int((df["Readiness_Level"]=="Medium").sum())
        low  = int((df["Readiness_Level"]=="Low").sum())
        none = int((df["Readiness_Level"]=="None").sum())
        adop = int(df["Adoption_TestBacked"].sum())
        intru = int((df[col_tests]==True).sum())
        print(f"  Readiness: High={high} ({high/N:.2%})  Medium={med} ({med/N:.2%})  Low={low} ({low/N:.2%})  None={none} ({none/N:.2%})")
        print(f"  Adoption (test-backed): {adop} / {intru} ({(adop/max(intru,1)):.2%} of androidTest repos)")
    except Exception:
        pass

if __name__ == "__main__":
    main()


[OK] Updated file in place: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet\3.2_Total_Repo.csv
  Readiness: High=437 (9.67%)  Medium=1903 (42.12%)  Low=9 (0.20%)  None=2169 (48.01%)
  Adoption (test-backed): 362 / 1770 (20.45% of androidTest repos)


In [9]:
# -*- coding: utf-8 -*-
"""
Add Automation_Level & Automation_Level_Strict with improved Android CI detection.

Changes in this version:
- Saves results by OVERWRITING the input file (e.g., 3.2_Total_Repo.csv).
- Still writes Automation_Level__counts.csv as a sidecar summary.

Inclusive Android CI inference:
  * Uses instru_t_ci_signal
  * Accepts Flutter CI on Android when devices include 'android' OR the runtime trail is Android-ish
  * Accepts direct runtime evidence (gradle connectedAndroidTest/deviceCheck, GMD, gcloud firebase, flank,
    browserstack, saucectl, appcenter test, maestro, adb am instrument, emulator setup)
Explicit FLIP:
  * True -> False only if flutter devices explicitly non-Android AND runtime trail lacks Android keywords
    after removing the generic 'boosted (AndroidTest present)' marker.

Input folder:
C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet
Prefers 3.2_Total_Repo.csv, else first CSV/XLSX.
"""

import re
import pandas as pd
from pathlib import Path

# -------- CONFIG --------
MAIN_DIR = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet"
PREFERRED_FILENAME = "3.2_Total_Repo.csv"

# -------- HELPERS --------
def find_main_file(folder: str) -> Path:
    folder = Path(folder)
    for pattern in [PREFERRED_FILENAME, "*.csv", "*.xlsx"]:
        matches = sorted(folder.glob(pattern))
        if matches:
            return matches[0]
    raise FileNotFoundError(f"No CSV/XLSX found in: {folder}")

def read_table(path: Path) -> pd.DataFrame:
    if path.suffix.lower() == ".csv":
        try:
            return pd.read_csv(path, low_memory=False)
        except UnicodeDecodeError:
            return pd.read_csv(path, encoding="latin-1", low_memory=False)
    return pd.read_excel(path, sheet_name=0)

def write_table(df: pd.DataFrame, path: Path) -> None:
    # OVERWRITE the input file, preserving its format
    if path.suffix.lower() == ".csv":
        df.to_csv(path, index=False)
    else:
        df.to_excel(path, index=False)

def to_bool_series(s: pd.Series) -> pd.Series:
    return s.astype(str).str.strip().str.lower().isin({"1","true","t","yes","y"})

def get_col(df: pd.DataFrame, names):
    for n in ([names] if isinstance(names, str) else names):
        if n in df.columns:
            return n
    return None

# --- Runtime keyword logic (strip generic boost first) ---
RX_AT_BOOST = re.compile(r'boosted\s*\(\s*androidtest\s+present\s*\)', re.I)
RX_RUNTIME = re.compile(
    r'(?:gcloud\s+firebase|flank|saucectl|appcenter\s+test|maestro|browserstack|bstack|'
    r'connected(?:android)?test|connectedcheck|devicecheck|'
    r'(?:^|[\s:/.-])(?:gradlew?|gradle)\b|'
    r'\bam\s+instrument\b|\bandroidtest\b|managed(?:virtual)?device|\bgmd\b|'
    r'reactivecircus/android-emulator-runner|sdkmanager[^"\n]*system-images|\bavdmanager\b)',
    re.I
)

def has_android_runtime(reason: str) -> bool:
    r = RX_AT_BOOST.sub("", str(reason or ""))
    return bool(RX_RUNTIME.search(r))

# --- Flutter device helpers ---
NULS = {"", "nan", "none", "null"}
SEP_RE = re.compile(r"[,\s;/|/]+")

def devices_nonblank_and_nonandroid(cell) -> bool:
    if pd.isna(cell): return False
    s = str(cell).strip().lower()
    if s in NULS: return False
    toks = [t for t in SEP_RE.split(s) if t]
    if not toks: return False
    return not any(("android" in t) or ("andorid" in t) for t in toks)

def has_android_in_devices(cell) -> bool:
    if pd.isna(cell): return False
    s = str(cell).strip().lower()
    return ("android" in s) or ("andorid" in s)

# -------- CORE COMPUTATION --------
def compute_labels(df: pd.DataFrame) -> pd.DataFrame:
    # Column names (robust fallbacks where helpful)
    col_ci     = get_col(df, ["instru_t_ci_signal", "yaml_pred", "ci_signal"])
    col_build  = get_col(df, ["instru_t_signal_config", "build_pred"])
    col_tests  = get_col(df, ["Intru_test", "androidtest_present", "at_pred"])
    col_fsig   = get_col(df, ["flutter_integ_t_signal", "flutter_cli_signal", "flutter_signal"])
    col_fdev   = get_col(df, ["flutter_integ_t_d", "flutter_devices", "flutter_device_list"])
    col_fcode  = get_col(df, ["flutter_instru_test"])
    col_env    = get_col(df, ["Android_Env_Family"])
    col_conf   = get_col(df, ["instru_t_ci_confidence"])
    col_reason = get_col(df, ["confidence_reason", "ci_confidence_reason", "ci_reason"])

    if not all([col_ci, col_build, col_tests]):
        missing = [n for n,c in [("instru_t_ci_signal",col_ci),
                                 ("instru_t_signal_config",col_build),
                                 ("Intru_test",col_tests)] if c is None]
        raise KeyError(f"Missing required columns: {missing}")

    # Base booleans
    ci_flag            = to_bool_series(df[col_ci])
    build_instru       = to_bool_series(df[col_build])
    test_code_present  = df[col_tests].astype(str).str.strip().str.lower().isin({"1","true","t","yes","y"})
    if col_fcode:
        test_code_present = test_code_present | to_bool_series(df[col_fcode])

    # Inclusive Android CI inference
    flutter_ci = to_bool_series(df[col_fsig]) if col_fsig else pd.Series([False]*len(df))
    flutter_on_android = df[col_fdev].apply(has_android_in_devices) if col_fdev else pd.Series([False]*len(df))
    reason_has_runtime = df[col_reason].apply(has_android_runtime) if col_reason else pd.Series([False]*len(df))

    ci_android_signal = (
        ci_flag |
        (flutter_ci & (flutter_on_android | reason_has_runtime)) |
        reason_has_runtime
    )

    # Explicit FLIP True->False (only when devices are explicitly non-Android AND no runtime keywords)
    if col_fdev:
        nonblank_nonandroid = df[col_fdev].apply(devices_nonblank_and_nonandroid)
    else:
        nonblank_nonandroid = pd.Series([False]*len(df))
    no_runtime_keywords = ~reason_has_runtime
    flip = nonblank_nonandroid & no_runtime_keywords
    ci_android_signal = ci_android_signal & (~flip)

    # Labels
    df = df.copy()
    df["ci_android_signal"]   = ci_android_signal
    df["build_instru_signal"] = build_instru
    df["test_code_present"]   = test_code_present

    def classify(row):
        ci, build, test = row["ci_android_signal"], row["build_instru_signal"], row["test_code_present"]
        if ci and build and test:               return "Fully_Automated"
        if (not ci) and build and test:         return "Semi_Automated"
        if ci and not (build and test):         return "CI_only_or_Incomplete"
        if (not ci) and (not build) and test:   return "Code_only_not_wired"
        return "No_Signal"

    df["Automation_Level"] = df.apply(classify, axis=1)

    # Strict upgrade (optional)
    if col_env:
        env_ok = df[col_env].astype(str).str.lower().isin(
            {"emulator","gmd","managedvirtualdevice","3p","3p lab","firebase",
             "browserstack","saucelabs","appcenter","real device","device farm"}
        )
    else:
        env_ok = pd.Series([True]*len(df))

    if col_conf:
        conf_ok = df[col_conf].astype(str).str.lower().isin({"high","medium"})
    else:
        conf_ok = pd.Series([True]*len(df))

    def classify_strict(row):
        if row["Automation_Level"] == "Fully_Automated":
            if env_ok.iloc[row.name] and conf_ok.iloc[row.name]:
                return "Fully_Automated_Strict"
        return row["Automation_Level"]

    df["Automation_Level_Strict"] = df.apply(classify_strict, axis=1)
    return df

def main():
    in_path = find_main_file(MAIN_DIR)
    print(f"[INFO] Using main spreadsheet: {in_path}")

    df = read_table(in_path)
    out_df = compute_labels(df)

    # === OVERWRITE the input file ===
    write_table(out_df, in_path)
    print(f"[OK] Overwrote: {in_path}")

    # Sidecar summary for quick QA
    counts = out_df["Automation_Level"].value_counts(dropna=False).rename_axis("Automation_Level").reset_index(name="count")
    counts_path = in_path.with_name("Automation_Level__counts.csv")
    counts.to_csv(counts_path, index=False)
    print(f"[OK] Wrote: {counts_path}")
    print(counts)

if __name__ == "__main__":
    main()


[INFO] Using main spreadsheet: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet\3.2_Total_Repo.csv
[OK] Overwrote: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet\3.2_Total_Repo.csv
[OK] Wrote: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet\Automation_Level__counts.csv
        Automation_Level  count
0              No_Signal   2673
1         Semi_Automated   1180
2        Fully_Automated    332
3    Code_only_not_wired    228
4  CI_only_or_Incomplete    105


In [10]:
# -*- coding: utf-8 -*-
"""
Adds `Instru_t_Adopted` (True/False) to the main spreadsheet per F1 (Adoption):
    Instru_t_Adopted := test_code_present AND (build_instru_signal OR ci_android_signal)

Where:
- test_code_present := (Intru_test == 1) OR (flutter_instru_test == True if present)
- build_instru_signal := truthy(instru_t_signal_config)
- ci_android_signal := inclusive CI inference (YAML + Flutter + runtime cues) with FLIP safeguard:
      ci_android_signal_pre := instru_t_ci_signal
                               OR (flutter_integ_t_signal AND (flutter_integ_t_d has 'android'/'andorid' OR confidence_reason has Android runtime cues))
                               OR (confidence_reason has Android runtime cues)
      FLIP True->False only if flutter_integ_t_d is non-blank and explicitly non-Android AND confidence_reason lacks runtime cues.

Input:
    C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet\3.2_Total_Repo.csv
Output:
    Overwrites the same CSV in place (creates a one-time backup alongside).
"""

import re
import pandas as pd
from pathlib import Path
from datetime import datetime

# -------- PATHS --------
MAIN_DIR = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet")
FILENAME = "3.2_Total_Repo.csv"
IN_PATH = MAIN_DIR / FILENAME

# -------- HELPERS --------
TRUTHY = {"1","true","t","yes","y"}

def to_bool_series(s: pd.Series) -> pd.Series:
    return s.astype(str).str.strip().str.lower().isin(TRUTHY)

def get_col(df: pd.DataFrame, names):
    """Return the first existing column name from a list (or the same str)."""
    for n in ([names] if isinstance(names, str) else names):
        if n in df.columns:
            return n
    return None

# Strip the generic boost before runtime parsing
RX_AT_BOOST = re.compile(r'boosted\s*\(\s*androidtest\s+present\s*\)', re.I)

# Android runtime cues (gradle tasks, GMD, 3P labs, emulator setup, adb, literal androidtest)
RX_RUNTIME = re.compile(
    r'(?:gcloud\s+firebase|flank|saucectl|appcenter\s+test|maestro|browserstack|bstack|'
    r'connected(?:android)?test|connectedcheck|devicecheck|'
    r'(?:^|[\s:/.-])(?:gradlew?|gradle)\b|'
    r'\bam\s+instrument\b|\bandroidtest\b|managed(?:virtual)?device|\bgmd\b|'
    r'reactivecircus/android-emulator-runner|sdkmanager[^"\n]*system-images|\bavdmanager\b)',
    re.I
)

def has_android_runtime(reason: str) -> bool:
    r = RX_AT_BOOST.sub("", str(reason or ""))
    return bool(RX_RUNTIME.search(r))

NULS = {"", "nan", "none", "null"}
SEP_RE = re.compile(r"[,\s;/|/]+")

def flutter_devices_nonblank_and_nonandroid(cell) -> bool:
    if pd.isna(cell): return False
    s = str(cell).strip().lower()
    if s in NULS: return False
    toks = [t for t in SEP_RE.split(s) if t]
    if not toks: return False
    return not any(("android" in t) or ("andorid" in t) for t in toks)

def flutter_devices_has_android(cell) -> bool:
    if pd.isna(cell): return False
    s = str(cell).strip().lower()
    return ("android" in s) or ("andorid" in s)

# -------- MAIN --------
def main():
    if not IN_PATH.exists():
        raise FileNotFoundError(f"Input not found: {IN_PATH}")

    df = pd.read_csv(IN_PATH, low_memory=False)

    # Resolve columns
    col_ci      = get_col(df, ["instru_t_ci_signal", "yaml_pred", "ci_signal"])
    col_build   = get_col(df, ["instru_t_signal_config", "build_pred"])
    col_tests   = get_col(df, ["Intru_test", "androidtest_present", "at_pred"])
    col_fsig    = get_col(df, ["flutter_integ_t_signal", "flutter_cli_signal", "flutter_signal"])
    col_fdev    = get_col(df, ["flutter_integ_t_d", "flutter_devices", "flutter_device_list"])
    col_fcode   = get_col(df, ["flutter_instru_test"])
    col_reason  = get_col(df, ["confidence_reason", "ci_confidence_reason", "ci_reason"])

    # Basic guards
    missing = [name for name, col in [("instru_t_ci_signal", col_ci),
                                      ("instru_t_signal_config", col_build),
                                      ("Intru_test", col_tests)] if col is None]
    if missing:
        raise KeyError(f"Missing required columns: {missing}")

    # Core booleans
    test_code_present = df[col_tests].astype(str).str.strip().str.lower().isin(TRUTHY)
    if col_fcode:
        test_code_present = test_code_present | to_bool_series(df[col_fcode])

    build_instru_signal = to_bool_series(df[col_build])

    # Inclusive CI inference
    ci_flag = to_bool_series(df[col_ci])

    flutter_ci = to_bool_series(df[col_fsig]) if col_fsig else pd.Series([False]*len(df))
    flutter_on_android = df[col_fdev].apply(flutter_devices_has_android) if col_fdev else pd.Series([False]*len(df))
    reason_has_runtime = df[col_reason].apply(has_android_runtime) if col_reason else pd.Series([False]*len(df))

    ci_android_signal_pre = ci_flag | (flutter_ci & (flutter_on_android | reason_has_runtime)) | reason_has_runtime

    # FLIP True->False only when devices explicitly non-Android AND no runtime cues
    nonblank_nonandroid = df[col_fdev].apply(flutter_devices_nonblank_and_nonandroid) if col_fdev else pd.Series([False]*len(df))
    flip = nonblank_nonandroid & (~reason_has_runtime)
    ci_android_signal = ci_android_signal_pre & (~flip)

    # F1: Instru_t_Adopted
    df["Instru_t_Adopted"] = (test_code_present & (build_instru_signal | ci_android_signal))

    # Save (overwrite), but write a one-time timestamped backup first
    backup = IN_PATH.with_name(IN_PATH.stem + f"__backup_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv")
    df.to_csv(backup, index=False)
    df.to_csv(IN_PATH, index=False)
    print(f"[OK] Added Instru_t_Adopted and overwrote: {IN_PATH}")
    print(f"[Backup] Wrote: {backup}")

if __name__ == "__main__":
    main()


[OK] Added Instru_t_Adopted and overwrote: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet\3.2_Total_Repo.csv
[Backup] Wrote: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet\3.2_Total_Repo__backup_20250901_182727.csv
